# Support vector machines: margins, geometry, and kernels

**Question.** Does a wide-margin classifier offer useful cultivar separation, and when does nonlinear geometry earn its complexity?

A support vector machine learns from points near a decision boundary. We will inspect feature geometry, define the margin and hinge loss, compare linear and radial-basis reference models through the package, and examine decision margins rather than treating an SVM score as a probability.

**What this example models.** Geometry, scaling, regularization, and kernel complexity are connected to an applied separation question and to the meaning of errors near the boundary.

**Your project version.** Show why a margin model is plausible for your features, implement and test the relevant optimization behavior, compare linear and nonlinear boundaries fairly, inspect support cases and errors, and decide whether added geometric complexity improves the real task.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from dsm_project.config import load_config
from dsm_project.data import load_project_dataset
from dsm_project.evaluation import comparison_table
from dsm_project.experiments import fit_supervised_candidate

config = load_config("configs/example.toml")
dataset = load_project_dataset(config)
task = dataset.tasks["classification"]
frame = dataset.observations

for label, group in frame.groupby(task.target_column):
    plt.scatter(group["proline"], group["color_intensity"], label=f"class {label}", alpha=0.75)
plt.xlabel("proline")
plt.ylabel("color intensity")
plt.title("Two coordinates reveal scale and overlap—not the full boundary")
plt.legend()
plt.tight_layout();

## Margin geometry

A linear decision score is $f(x)=w^\top x+b$. In the binary case the boundary is the hyperplane $f(x)=0$, and the geometric distance to it is $|f(x)|/\lVert w\rVert_2$. A soft-margin SVM balances a wide margin against violations through an objective of the form

$$
\frac{1}{2}\lVert w\rVert_2^2
+C\sum_i\max(0,1-y_i f(x_i)).
$$

The hinge term is zero only when the signed score clears the unit margin. Feature scaling changes distances, dot products, regularization, and the meaning of $C$, so scaling belongs inside the fitted pipeline.

A kernel replaces inner products with a similarity function. This can express nonlinear boundaries without explicitly constructing every transformed coordinate, but it changes hyperparameter selection, interpretability, memory, and latency.

In [ ]:
table = comparison_table(
    dataset,
    config,
    task_name="classification",
    model_names=("linear_svm", "kernel_svm"),
)
display(table)

linear = fit_supervised_candidate(
    dataset, config, task_name="classification", model_name="linear_svm"
)
kernel = fit_supervised_candidate(
    dataset, config, task_name="classification", model_name="kernel_svm"
)
print("Linear support vectors per class:", linear.estimator.named_steps["svc"].n_support_)
print("Kernel support vectors per class:", kernel.estimator.named_steps["svc"].n_support_)

## Inspect the decision margin

The largest class score is not a calibrated probability. The gap between the two largest scores is a useful diagnostic of ambiguity under this fitted model: small gaps indicate that two class decisions are close. It still does not measure all forms of uncertainty or distance from the training distribution.

In [ ]:
scores = kernel.estimator.decision_function(kernel.holdout_features)
sorted_scores = np.sort(scores, axis=1)
score_gap = sorted_scores[:, -1] - sorted_scores[:, -2]
correct = kernel.predictions == kernel.observed

plt.figure(figsize=(8, 4))
plt.hist(score_gap[correct], bins=10, alpha=0.7, label="correct", color="#1b9e77")
plt.hist(score_gap[~correct], bins=10, alpha=0.7, label="incorrect", color="#d95f02")
plt.xlabel("gap between two largest decision scores")
plt.ylabel("holdout count")
plt.title("Small score gaps identify ambiguous fitted decisions")
plt.legend()
plt.tight_layout();

print("Linear balanced accuracy:", linear.metric_value)
print("Kernel balanced accuracy:", kernel.metric_value)

## Conclusion, limits, and student implementation

A kernel is not automatically better because it is nonlinear. Compare it with a scaled linear margin model using validation evidence, inspect the number of support vectors, and measure prediction resources. On small data, apparent differences can be mostly split variation.

A student linear-SVM implementation should expose its score, regularization and optimization conventions; test hand-computable margin and hinge cases; reject incompatible labels and shapes; and compare against a trusted solver only when objectives match. Document kernel selection and parameter search as model-selection decisions, never as information learned from the final test set.